In [1]:
import sys
import pandas as pd
from pathlib import Path
from linearmodels.panel import PanelOLS
sys.path.append(str(Path.cwd().parent))
from constants import PATH_TO_FINAL_OUTPUT
import statsmodels.api as sm


In [2]:
df_source = pd.read_csv(PATH_TO_FINAL_OUTPUT)
df_speaker_points = df_source.copy()


In [3]:
df_speaker_points['debate_date'] = pd.to_datetime(df_speaker_points['debate_date'])
df_speaker_points['speaker_first_debate_date'] = pd.to_datetime(df_speaker_points['speaker_first_debate_date'])

motion_balance = df_speaker_points[df_speaker_points['side'] == 'aff'].groupby(['debate_id', 'motion'])['ballots_gained'].mean().reset_index()
motion_balance = motion_balance.groupby('motion')['ballots_gained'].mean().reset_index()
motion_balance.columns = ['motion', 'motion_balance']

df_speaker_points = df_speaker_points.merge(motion_balance, on='motion', how='left')

df_speaker_points['years_since_first_debate'] = df_speaker_points['debate_date'].dt.year - df_speaker_points['speaker_first_debate_date'].dt.year

df_speaker_points['tournament_round'] = df_speaker_points.groupby('tournament_id')['debate_date'].rank(method='dense').astype(int)

df_speaker_points['is_aff'] = (df_speaker_points['side'] == 'aff').astype(int)
df_speaker_points['motion_balance_x_aff'] = df_speaker_points['motion_balance'] * df_speaker_points['is_aff']

In [4]:
def get_teammate_avg(group):
    teammate_avgs = []
    for idx in group.index:
        other_scores = group.loc[group.index != idx, 'speaker_points']
        teammate_avgs.append(other_scores.mean())
    return pd.Series(teammate_avgs, index=group.index)

df_speaker_points['avg_teammate_score'] = df_speaker_points.groupby(['debate_id', 'side'], group_keys=False).apply(get_teammate_avg)

In [5]:
df_reg = df_speaker_points.dropna(subset=['speaker_points', 'is_male', 'years_since_first_debate', 
                            'tournament_round', 'motion_balance_x_aff', 'motion_balance', 
                            'speaker_name', 'avg_teammate_score'])

X_pooled = df_reg[['is_male', 'years_since_first_debate', 'tournament_round', 
                    'motion_balance_x_aff', 'motion_balance', 'avg_teammate_score']].astype(float)
y_pooled = df_reg['speaker_points'].astype(float)

X_pooled = sm.add_constant(X_pooled)
model_pooled = sm.OLS(y_pooled, X_pooled).fit()

In [6]:
df_panel = df_reg.copy()
df_panel['speaker_id'] = pd.Categorical(df_panel['speaker_name']).codes
df_panel = df_panel.set_index(['speaker_id', 'debate_date'])

y_panel = df_panel['speaker_points']
# we have to drop years since first debate and is_male to use the fixed effects model
X_panel = df_panel[['tournament_round', 
                     'motion_balance_x_aff', 'motion_balance', 'avg_teammate_score']].astype(float)


model_fixed_effects = PanelOLS(y_panel, X_panel, entity_effects=True).fit()

In [7]:
print("="*80)
print("Pooled OLS Regression Results")
print("="*80)
print(model_pooled.summary())

print("\n" + "="*80)
print("Fixed Effects Panel Regression Results (Speaker Fixed Effects)")
print("="*80)
print(model_fixed_effects.summary)


print("\n" + "="*80)
print("Model Comparison")
print("="*80)
print(f"Pooled OLS R-squared: {model_pooled.rsquared:.4f}")
print(f"Fixed Effects R-squared: {model_fixed_effects.rsquared:.4f}")
print(f"Fixed Effects R-squared (within): {model_fixed_effects.rsquared_within:.4f}")

Pooled OLS Regression Results
                            OLS Regression Results                            
Dep. Variable:         speaker_points   R-squared:                       0.586
Model:                            OLS   Adj. R-squared:                  0.582
Method:                 Least Squares   F-statistic:                     155.8
Date:                Thu, 22 Jan 2026   Prob (F-statistic):          6.62e-123
Time:                        21:08:56   Log-Likelihood:                -1695.9
No. Observations:                 667   AIC:                             3406.
Df Residuals:                     660   BIC:                             3437.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [8]:
print("\n" + "="*80)
print("PART 2: Team-Level Analysis")
print("="*80)


PART 2: Team-Level Analysis


In [9]:
df_team = df_reg.copy()

df_team['prop_male'] = df_team.groupby(['debate_id', 'side'])['is_male'].transform('mean')

df_team['avg_team_experience'] = df_team.groupby(['debate_id', 'side'])['years_since_first_debate'].transform('mean')

team_debate = df_team.groupby(['debate_id', 'side']).agg({
    'ballots_gained': 'first',
    'prop_male': 'first',
    'avg_team_experience': 'first',
    'speaker_name': 'count'
}).reset_index()

team_debate.columns = ['debate_id', 'side', 'ballots_gained', 'prop_male', 'avg_team_experience', 'team_size']

team_stats = team_debate.groupby('side').agg({
    'ballots_gained': 'sum',
    'debate_id': 'count'
}).reset_index()

team_stats.columns = ['side', 'total_ballots', 'num_debates']
team_debate = team_debate.merge(team_stats, on='side', how='left')

team_debate['ballots_per_debate'] = team_debate['total_ballots'] / team_debate['num_debates'] / 3

team_debate_clean = team_debate.dropna(subset=['ballots_per_debate', 'prop_male', 'avg_team_experience'])


In [10]:
X_team = team_debate_clean[['prop_male', 'avg_team_experience']].astype(float)
y_team = team_debate_clean['ballots_per_debate'].astype(float)

X_team = sm.add_constant(X_team)
model_team = sm.OLS(y_team, X_team).fit()

print("\n" + "="*80)
print("Team-Level OLS Regression Results")
print("DV: Proportion of Debates Won Decisively (ballots_gained / num_debates / 3)")
print("="*80)
print(model_team.summary())


Team-Level OLS Regression Results
DV: Proportion of Debates Won Decisively (ballots_gained / num_debates / 3)
                            OLS Regression Results                            
Dep. Variable:     ballots_per_debate   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.003
Method:                 Least Squares   F-statistic:                    0.2245
Date:                Thu, 22 Jan 2026   Prob (F-statistic):              0.799
Time:                        21:09:03   Log-Likelihood:                 913.34
No. Observations:                 506   AIC:                            -1821.
Df Residuals:                     503   BIC:                            -1808.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
-----------